In [24]:
# graphrag_plus.py

import json
import pandas as pd
import networkx as nx

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import ollama



In [25]:
# Load FAQ (already generated from refined.csv using your script)
faq = pd.read_csv("faq.csv", encoding="utf-8")

# Load labels JSONL (intent + entities)
labels = []
with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        labels.append(json.loads(line))

print(f"FAQ rows: {len(faq)}")
print(f"Labelled items: {len(labels)}")


FAQ rows: 13
Labelled items: 111


In [26]:
import json

with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    labels = [json.loads(line) for line in f]

# Extract all unique intents
unique_intents = set()
for item in labels:
    for intent in item.get("labels", {}).get("intents", []):
        unique_intents.add(intent)

print("Unique Intents:", unique_intents)


Unique Intents: {'send_materials', 'confirm', 'share_feedback', 'request_feedback', 'schedule', 'request_info', 'accept_or_decline', 'follow_up', 'reschedule'}


In [27]:
import json

with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i > 4: break
        print(json.loads(line))


{'id': 'a1bc948c-4475-4b2f-ada8-84de14a52a57', 'labels': {'topic': 'Professor/Academic', 'intents': ['accept_or_decline', 'share_feedback'], 'artifacts': ['offer_letter']}, 'subject': 'Offer for RA Position in Our Lab', 'sender_email': 'jane.smith@university.edu'}
{'id': '406b8fae-4c32-4b7f-bfdf-ba5642bdd466', 'labels': {'topic': 'Group/Event Coordination', 'intents': ['accept_or_decline', 'reschedule', 'request_info'], 'artifacts': []}, 'subject': 'Team Meeting Availability Confirmation', 'sender_email': 'doe.jane@example.com'}
{'id': '9180c14f-f779-4339-8495-eb765e329618', 'labels': {'topic': 'Professor/Academic', 'intents': ['share_feedback', 'request_info', 'request_feedback'], 'artifacts': ['evaluation_sheet', 'application_form', 'draft']}, 'subject': 'Discussion about Recent Submission', 'sender_email': 'drsmith@university.edu'}
{'id': '2bd37763-52d8-4d0e-a872-1208620fc1bc', 'labels': {'topic': 'Feedback & Reviews', 'intents': ['request_info', 'reschedule'], 'artifacts': ['applic

In [28]:
import networkx as nx

G = nx.DiGraph()

for item in labels:
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])

    if not topic and not intents and not artifacts:
        continue

    # Add topic node
    if topic:
        G.add_node(topic, type="topic")

    # Add intents as nodes and edges
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            G.add_edge(topic, intent, relation="HAS_INTENT")

    # Add artifacts as nodes and edges
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            G.add_edge(topic, artifact, relation="USES_ARTIFACT")

print(f"Graph: {len(G.nodes())} nodes, {len(G.edges())} edges")


Graph: 31 nodes, 99 edges


In [29]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

qdrant = QdrantClient(path="qdrant_data")   # persistent local storage

qdrant.recreate_collection(
    collection_name="knowledge_space",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


True

## if the above cell doesn't work , RUN rm -f qdrant_data/.lock      

In [30]:
faq_texts = [
    f"FAQ | Question: {row['question']} | Answer: {row['answer']}"
    for _, row in faq.iterrows()
]
faq_vectors = embedder.encode(faq_texts, show_progress_bar=False)
faq_payloads = []

for idx, row in faq.iterrows():
    faq_payloads.append({
        "type": "faq",
        "id_kind": "faq",
        "faq_id": int(idx),
        "question": row["question"], 
        "answer": row["answer"],
    })


In [31]:
graph_nodes = list(G.nodes(data=True))  # [(name, attrs), ...]

graph_texts = []
graph_payloads = []

for i, (name, attrs) in enumerate(graph_nodes):
    ntype = attrs.get("type", "unknown")
    neighbors = list(G.successors(name)) + list(G.predecessors(name))
    neighbors_str = ", ".join(neighbors) if neighbors else "None"

    text = f"GRAPH_NODE | Type: {ntype} | Name: {name} | Neighbors: {neighbors_str}"
    graph_texts.append(text)

    graph_payloads.append({
        "type": "graph_node",
        "id_kind": "graph_node",
        "node_name": name,
        "node_type": ntype,
        "neighbors": neighbors,
    })

graph_vectors = embedder.encode(graph_texts, show_progress_bar=False)


In [32]:
points = []

# FAQ points
for i, (vec, payload) in enumerate(zip(faq_vectors, faq_payloads)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload=payload
        )
    )

offset = len(points)

# Graph-node points (id continues after FAQ)
for j, (vec, payload) in enumerate(zip(graph_vectors, graph_payloads)):
    points.append(
        PointStruct(
            id=offset + j,
            vector=vec.tolist(),
            payload=payload
        )
    )

qdrant.upsert(collection_name="knowledge_space", points=points)
print(f"✅ Upserted {len(points)} total points into 'knowledge_space'")


✅ Upserted 44 total points into 'knowledge_space'


In [43]:
# Close the Qdrant client to release the lock
qdrant = None
import gc
gc.collect()

2764

In [44]:
# Release Qdrant connection
qdrant = None
import gc
gc.collect()
print("✅ Qdrant released - backend can now start")

✅ Qdrant released - backend can now start


In [33]:
def simple_intent_classifier(email_text: str) -> str:
    text = email_text.lower()
    if "refund" in text or "charged" in text:
        return "Refund Request"
    if "meeting" in text or "call" in text:
        return "Meeting Request"
    return "General Inquiry"


In [34]:
import ollama
import json

def classify_intent_llm(email_text: str, available_intents: list):
    """
    Use LLM (Ollama) to classify the intent based on email text and your labeled intent list.
    """
    prompt = f"""
    You are an intent classification assistant.
    Below is a list of valid intents extracted from training data:
    {available_intents}

    Read the email carefully and return ONLY a JSON object with the most relevant intent.
    If multiple intents fit, return the one that best describes the user's main goal.

    Email:
    \"\"\"{email_text}\"\"\"

    Output example:
    {{"intent": "request_feedback"}}
    """

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        content = response["message"]["content"]
        parsed = json.loads(content)
        return parsed.get("intent", "general_inquiry")
    except Exception:
        return "general_inquiry"


In [35]:
def build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info):
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"

    graph_section = "\n".join([
        f"{i+1}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}"
        for i, g in enumerate(graph_hits)
    ]) or "None"

    expansion_section = "\n".join([
        f"{i+1}. {node} → {neighbors}"
        for i, (node, neighbors) in enumerate(expanded_graph_info.items())
    ]) or "None"

    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intent**: {intent}

📘 **Relevant FAQs**
{faq_section}

🧩 **Graph Context**
{graph_section}

🔗 **Related Concepts**
{expansion_section}

---

Write your reply in Zubair’s tone:
- Acknowledge the sender and context.
- If an action is requested, confirm or ask a polite follow-up question.
- Keep the reply under 120 words.
- Do NOT invent facts — only use what’s in context.
"""
    return prompt


In [ ]:
"""
def build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info):
    faq_section = ""
    for i, f in enumerate(faq_hits, 1):
        faq_section += f"{i}. Q: {f['question']}\n   A: {f['answer']}\n"

    graph_section = ""
    for i, g in enumerate(graph_hits, 1):
        graph_section += f"{i}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}\n"

    expansion_section = ""
    for i, (node, neighbors) in enumerate(expanded_graph_info.items(), 1):
        expansion_section += f"{i}. {node} → {neighbors}\n"

    prompt = f
You are a helpful student copilot.

USER EMAIL:
\"\"\"{email_text}\"\"\"

DETECTED INTENT: {intent}

FACTUAL CONTEXT (from FAQ):
{faq_section if faq_section else "None"}

GRAPH CONTEXT (graph nodes hit by retrieval):
{graph_section if graph_section else "None"}

GRAPH EXPANSION (neighbors of important nodes):
{expansion_section if expansion_section else "None"}

Using ONLY the information above, write a polite, accurate, and concise reply to the user.
If some detail is not present, do NOT invent facts.

    return prompt

"""

In [36]:
def answer_email(email_text: str, top_k: int = 6, show_context: bool = True):
    # Intent classification (placeholder)
    intent = classify_intent_llm(email_text, list(unique_intents))


    # Embed query
    q_vec = embedder.encode([email_text])[0].tolist()

    # Qdrant unified retrieval
    hits = qdrant.query_points(
    collection_name="knowledge_space",
    query=q_vec,   
    limit=top_k
).points


    faq_hits, graph_hits = [], []

    # Separate by type
    for h in hits:
        p = h.payload
        if p["type"] == "faq":
            faq_hits.append({"score": h.score, **p})
        elif p["type"] == "graph_node":
            graph_hits.append({"score": h.score, **p})

    # Graph expansion (for graph hits)
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print context (for debug)
    if show_context:
        print("\n🔍 Retrieved FAQ Chunks:")
        if faq_hits:
            for f in faq_hits:
                print(f"  [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"                A: {f['answer']}\n")
        else:
            print("  None\n")

        print("Retrieved Graph Nodes:")
        if graph_hits:
            for g in graph_hits:
                print(f"  [Score {g['score']:.3f}] Node: {g['node_name']} (Type: {g['node_type']})")
                print(f"                Neighbors: {g.get('neighbors', [])}\n")
        else:
            print("  None\n")

    # Build prompt
    prompt = build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info)

    # Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Confidence gating
    top_score = hits[0].score if hits else 0.0
    auto_send = top_score > 0.85

    # Return full result
    return {
        "intent": intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }


In [38]:
"""
Fixed answer_email function for graph_rag_updated.ipynb
Paste this into a new cell in your notebook and run it.
"""

def answer_email(email_text: str, top_k: int = 6, show_context: bool = True):
    # Intent classification (placeholder)
    intent = classify_intent_llm(email_text, list(unique_intents))

    # Embed query
    q_vec = embedder.encode([email_text])[0].tolist()

    # Qdrant unified retrieval - using search() for compatibility
    try:
        # Try newer API first
        hits = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,   
            limit=top_k
        ).points
    except AttributeError:
        # Fallback to older API (for qdrant-client < 1.7)
        hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k,
            with_payload=True
        )

    faq_hits, graph_hits = [], []

    # Separate by type
    for h in hits:
        p = h.payload
        if p["type"] == "faq":
            faq_hits.append({"score": h.score, **p})
        elif p["type"] == "graph_node":
            graph_hits.append({"score": h.score, **p})

    # Graph expansion (for graph hits)
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print context (for debug)
    if show_context:
        print("\n🔍 Retrieved FAQ Chunks:")
        if faq_hits:
            for f in faq_hits:
                print(f"  [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"                A: {f['answer']}\n")
        else:
            print("  None\n")

        print("Retrieved Graph Nodes:")
        if graph_hits:
            for g in graph_hits:
                print(f"  [Score {g['score']:.3f}] Node: {g['node_name']} (Type: {g['node_type']})")
                print(f"                Neighbors: {g.get('neighbors', [])}\n")
        else:
            print("  None\n")

    # Build prompt
    prompt = build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info)

    # Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Confidence gating
    top_score = hits[0].score if hits else 0.0
    auto_send = top_score > 0.85

    # Return full result
    return {
        "intent": intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }



In [41]:
"""
Enhanced answer_email function with multi-intent classification and graph retrieval.
"""

def answer_email_enhanced(email_text: str, top_k: int = 6, show_context: bool = True):
    """
    Enhanced email answering with:
    1. Multi-intent classification
    2. Vector search (FAQs)
    3. Intent-based graph retrieval
    4. Graph expansion
    """
    
    # Step 1: Multi-intent classification
    intents = classify_multi_intent(email_text, list(unique_intents))
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print(f"🎯 Detected Intents: {intents}")
        print(f"   Primary: {primary_intent}\n")
    
    # Step 2: Embed query for vector search
    q_vec = embedder.encode([email_text])[0].tolist()

    # Step 3: Vector search in Qdrant
    try:
        hits = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,   
            limit=top_k
        ).points
    except AttributeError:
        hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k,
            with_payload=True
        )

    # Step 4: Separate FAQ and graph hits from vector search
    faq_hits, graph_hits = [], []
    for h in hits:
        p = h.payload
        if p["type"] == "faq":
            faq_hits.append({"score": h.score, **p})
        elif p["type"] == "graph_node":
            graph_hits.append({"score": h.score, **p})

    if show_context:
        print(f"📊 Vector Search Results:")
        print(f"   FAQ hits: {len(faq_hits)}")
        print(f"   Graph hits from vector search: {len(graph_hits)}\n")

    # Step 5: Intent-based graph retrieval (NEW!)
    intent_graph_nodes = get_nodes_by_intents(intents, limit=5)
    
    # Convert to graph hit format and merge with vector search results
    for node in intent_graph_nodes:
        # Avoid duplicates
        if not any(g.get("node_name") == node["name"] for g in graph_hits):
            graph_hits.append({
                "score": 0.75,  # Assign reasonable score for intent-based matches
                "node_name": node["name"],
                "node_type": node["type"],
                "neighbors": node["neighbors"]
            })
    
    if show_context:
        print(f"🕸️ After Intent-based Graph Retrieval:")
        print(f"   Total graph nodes: {len(graph_hits)}\n")

    # Step 6: Graph expansion
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print detailed context
    if show_context:
        print("\n" + "="*70)
        print("🔍 RETRIEVED CONTEXT")
        print("="*70)
        
        print("\n📚 FAQ Chunks:")
        if faq_hits:
            for i, f in enumerate(faq_hits, 1):
                print(f"  {i}. [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"     A: {f['answer'][:80]}...\n")
        else:
            print("  None\n")

        print("🕸️ Graph Nodes:")
        if graph_hits:
            for i, g in enumerate(graph_hits, 1):
                print(f"  {i}. [Score {g['score']:.3f}] {g['node_name']} (type: {g['node_type']})")
                neighbors_str = ", ".join(g.get('neighbors', [])[:5])
                print(f"     Neighbors: {neighbors_str}\n")
        else:
            print("  None\n")
        
        print("🔗 Expanded Graph Context:")
        if expanded_graph_info:
            for node, neighbors in list(expanded_graph_info.items())[:5]:
                print(f"  {node} → {', '.join(neighbors[:5])}")
        else:
            print("  None")
        
        print("="*70 + "\n")

    # Step 7: Build enhanced prompt
    prompt = build_prompt(
        email_text, 
        ", ".join(intents),  # Pass all intents
        faq_hits, 
        graph_hits, 
        expanded_graph_info
    )

    # Step 8: Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Step 9: Confidence scoring
    top_score = hits[0].score if hits else 0.0
    auto_send = top_score > 0.85

    return {
        "intents": intents,
        "primary_intent": primary_intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }


def classify_multi_intent(email_text: str, available_intents: list) -> list:
    """
    Classify multiple intents in an email using LLM.
    Returns list of intents (can be multiple).
    """
    intents_str = ", ".join(available_intents)
    
    prompt = f"""You are an email intent classifier.
Given the following email, identify ALL relevant intents from this list: {intents_str}

An email can have multiple intents (e.g., someone asking to send materials AND schedule a meeting).

Email:
\"\"\"{email_text}\"\"\"

Respond with a JSON array of intent labels (use underscores for multi-word intents).
Example: ["send_materials", "request_info"]
If none match well, respond with ["general_inquiry"]."""

    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response["message"]["content"].strip()
        
        # Try to parse JSON
        import json
        try:
            intents = json.loads(content)
            if isinstance(intents, list):
                # Normalize intents
                normalized = []
                known_lower = [i.lower().replace(" ", "_") for i in available_intents]
                
                for intent in intents:
                    intent_clean = intent.lower().replace(" ", "_")
                    if intent_clean in known_lower:
                        normalized.append(intent_clean)
                
                if normalized:
                    return normalized
        except json.JSONDecodeError:
            # Fallback: treat as single intent
            intent = content.lower().replace(" ", "_").strip('[]"')
            known_lower = [i.lower().replace(" ", "_") for i in available_intents]
            if intent in known_lower:
                return [intent]
        
        return ["general_inquiry"]
        
    except Exception as e:
        print(f"⚠️ Intent classification error: {e}")
        return ["general_inquiry"]


def get_nodes_by_intents(intents: list, limit: int = 5) -> list:
    """
    Retrieve graph nodes related to the detected intents.
    This is the key enhancement - direct intent-to-node lookup!
    """
    nodes = []
    seen_names = set()
    
    for intent in intents:
        # Check if intent exists as a node in graph
        if intent in G:
            # Get node info
            node_data = G.nodes[intent]
            neighbors = list(G.successors(intent)) + list(G.predecessors(intent))
            
            if intent not in seen_names:
                nodes.append({
                    "name": intent,
                    "type": node_data.get("type", "unknown"),
                    "neighbors": neighbors
                })
                seen_names.add(intent)
            
            # Also get connected nodes (topics, artifacts)
            for neighbor in neighbors:
                if neighbor not in seen_names and len(nodes) < limit:
                    neighbor_data = G.nodes[neighbor]
                    neighbor_neighbors = list(G.successors(neighbor)) + list(G.predecessors(neighbor))
                    nodes.append({
                        "name": neighbor,
                        "type": neighbor_data.get("type", "unknown"),
                        "neighbors": neighbor_neighbors
                    })
                    seen_names.add(neighbor)
    
    return nodes[:limit]


print("✅ Enhanced answer_email function loaded!")
print("📝 Usage: result = answer_email_enhanced(email_text)")
print("🆕 New features:")
print("   - Multi-intent classification")
print("   - Intent-based graph retrieval")
print("   - Enhanced context display")



✅ Enhanced answer_email function loaded!
📝 Usage: result = answer_email_enhanced(email_text)
🆕 New features:
   - Multi-intent classification
   - Intent-based graph retrieval
   - Enhanced context display


In [42]:
test_email = """Hi Iram,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc."""

result = answer_email_enhanced(test_email)
print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)

🎯 Detected Intents: ['general_inquiry']
   Primary: general_inquiry

📊 Vector Search Results:
   FAQ hits: 6
   Graph hits from vector search: 0

🕸️ After Intent-based Graph Retrieval:
   Total graph nodes: 0


🔍 RETRIEVED CONTEXT

📚 FAQ Chunks:
  1. [Score 0.372] Q: Can you share your resume and portfolio?
     A: Sure, here are the links to my resume and portfolio: Resume - https://drive.goog...

  2. [Score 0.356] Q: What additional information has been provided by Zubair for his application?
     A: Zubair has attached previous evaluation forms and feedback reports, a summary of...

  3. [Score 0.329] Q: Where can I find Zubair's professional profiles?
     A: Zubair maintains a LinkedIn profile (https://www.linkedin.com/in/zubair-atha/) a...

  4. [Score 0.289] Q: Where can I find your LinkedIn profile?
     A: You can view my LinkedIn profile at https://www.linkedin.com/in/zubair-atha/...

  5. [Score 0.280] Q: What are some of the projects Zubair is currently involved in?
     A

### The top similarity score measures how semantically close your input is to your best-matching knowledge chunk. It’s a confidence proxy for deciding whether the LLM’s reply is likely grounded in the right retrieved context.

In [20]:
test_email = (
    "Hi Zubair, I hope you’re doing well. Can I get your linkedin profile to connect?"
)
result = answer_email(test_email)

print("Intent:", result["intent"])
print("Top Score:", result["top_score"])
print("Auto-send:", result["auto_send"])
print("\n------ DRAFT REPLY ------\n")
print(result["reply"])


🔍 Retrieved FAQ Chunks:
  [Score 0.668] Q: Where can I find your LinkedIn profile?
                A: You can view my LinkedIn profile at https://www.linkedin.com/in/zubair-atha/

  [Score 0.549] Q: Where can I find Zubair's professional profiles?
                A: Zubair maintains a LinkedIn profile (https://www.linkedin.com/in/zubair-atha/) and has a GitHub repository (https://github.com/zubairatha) where he shares his work. He also provides a resume via Google Drive.

  [Score 0.481] Q: What additional information has been provided by Zubair for his application?
                A: Zubair has attached previous evaluation forms and feedback reports, a summary of his research experience, and reattached the job description. He also shared his resume and LinkedIn profile link: https://www.linkedin.com/in/zubair-atha/

  [Score 0.345] Q: How can I contact Zubair regarding updates or new sections in a report?
                A: You can reach out to Zubair directly via email to share any 